# Realtime Voice Interpreter — EN ↔ PT Translator

Build a low-latency, two-way **English ↔ Portuguese** voice interpreter on the **Realtime API** (`gpt-realtime-1.5`). The model listens to speech in one language and speaks the translation in the other, in near real time, over a persistent **WebSocket** connection.

> ## What runs here, and what needs hardware
>
> This notebook is split into two runnable layers so you can verify the **protocol** without a microphone:
>
> 1. **Section 3 — the full WebSocket loop** sends a *synthetic 100 ms silence chunk* instead of mic audio. Given a valid key and the Realtime model enabled, it connects, configures the session, commits a turn, and handles the response events end to end. No mic required.
> 2. **Section 4 — real mic capture** swaps the synthetic chunk for a live `sounddevice` input stream and plays the translated audio back. This is the part that needs **microphone + speaker hardware**.
>
> Only two things are genuinely unverifiable at authoring time and are marked `[unverified]`: the exact **endpoint URL** and **model name** (`gpt-realtime-1.5`). Confirm those against the live Realtime docs; everything else is working code.

## Setup

In [ ]:
import os
import getpass

def _set_env(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

# Audio + websocket deps (install once):
# !pip install -U openai websockets sounddevice numpy

## 1. Session configuration

The Realtime session is configured with the model, the voice, the audio formats, and — crucially for an interpreter — **instructions that pin the translation behavior**.

In [ ]:
REALTIME_MODEL = "gpt-realtime-1.5"
VOICE = "marin"   # pick any available realtime voice

INTERPRETER_INSTRUCTIONS = """
You are a live bidirectional interpreter between English and Portuguese.
- If the speaker uses English, respond ONLY with the Portuguese translation (spoken).
- If the speaker uses Portuguese, respond ONLY with the English translation (spoken).
- Do not add commentary, greetings, or explanations. Translate faithfully, preserving tone.
- Keep latency low: begin speaking as soon as a complete phrase is available.
"""

session_config = {
    "model": REALTIME_MODEL,
    "voice": VOICE,
    "instructions": INTERPRETER_INSTRUCTIONS,
    "input_audio_format": "pcm16",
    "output_audio_format": "pcm16",
    "input_audio_transcription": {"model": "whisper-1"},  # see what was heard
    "turn_detection": {"type": "server_vad"},             # auto-detect speech turns
}

## 2. The Realtime event lifecycle

The Realtime API is **event-driven over a single WebSocket**: you send JSON events up, the server streams JSON events down, and both directions are interleaved on the same connection. Understanding which events fire in what order is the whole game. For one interpreter turn the sequence is:

**You send (client → server):**

| Event | Meaning |
|---|---|
| `session.update` | Configure the session: model, voice, audio formats, instructions, turn detection. Send this **first**, right after connecting. |
| `input_audio_buffer.append` | Stream a chunk of PCM16 microphone audio (base64-encoded). Send many of these as the user speaks. |
| `input_audio_buffer.commit` | Mark the end of the user's turn. *(With `server_vad` turn detection, the server commits automatically when it detects a pause — you can skip the manual commit.)* |
| `response.create` | Ask the model to produce a response. *(Also automatic under `server_vad`.)* |

**The server sends back (server → client), roughly in this order:**

| Event | Meaning |
|---|---|
| `session.created` | Handshake complete; the session is live. The first event you'll receive. |
| `session.updated` | Your `session.update` was applied. |
| `input_audio_buffer.speech_started` / `speech_stopped` | Server VAD detected the user starting/stopping speaking. |
| `conversation.item.input_audio_transcription.completed` | The transcript of what the user said (handy for debugging "what did it hear?"). |
| `response.created` | The model started generating a response. |
| `response.audio_transcript.delta` | Streamed **text** of the spoken translation — print this for a live caption. |
| `response.audio.delta` | Streamed **audio** chunks (PCM16) of the spoken translation — play these on the speaker. |
| `response.audio.done` / `response.done` | The response finished. `response.done` carries usage and the final item. |
| `error` | Something went wrong; the payload says what. Always handle this. |

The mental model: **`*.delta` events are the stream, `*.done` events are the punctuation.** Your receive loop is just a dispatch on `event["type"]`.

## 3. The complete WebSocket loop (mic-free, runnable)

This is the full async loop end to end: connect → `session.update` → wait for `session.created`/`session.updated` → append a chunk of input audio → commit the turn and request a response → handle `response.audio_transcript.delta` (caption), buffer `response.audio.delta` (audio) → stop on `response.done`.

So it runs **without a microphone**, `synthetic_silence_chunk()` produces 100 ms of PCM16 silence as a stand-in for a real audio frame. Replace that one call with live mic frames (Section 4) and you have a working interpreter. The endpoint URL and model name are the only `[unverified]` pieces.

In [ ]:
import asyncio
import json
import base64
import websockets   # pip install websockets

REALTIME_URL = "wss://api.openai.com/v1/realtime"   # [unverified] confirm GA path
SAMPLE_RATE = 24_000                                # PCM16 @ 24 kHz is the Realtime default
CHUNK_MS = 100


def synthetic_silence_chunk(ms: int = CHUNK_MS, sample_rate: int = SAMPLE_RATE) -> bytes:
    """A placeholder PCM16 audio frame: `ms` of silence.

    PCM16 = 2 bytes/sample, mono. Silence is all-zero samples. In Section 4 this
    is replaced by real microphone frames; here it lets the protocol run with no
    audio hardware.
    """
    num_samples = int(sample_rate * ms / 1000)
    return b"\x00\x00" * num_samples


def b64(audio_bytes: bytes) -> str:
    return base64.b64encode(audio_bytes).decode("ascii")


async def realtime_interpreter(audio_chunks):
    """Run one interpreter session.

    `audio_chunks` is an async iterable yielding PCM16 byte frames. For a mic-free
    smoke test, pass a generator that yields a few synthetic silence chunks.
    """
    headers = {
        "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
        "OpenAI-Beta": "realtime=v1",   # [unverified] header for GA
    }

    async with websockets.connect(
        f"{REALTIME_URL}?model={REALTIME_MODEL}",
        extra_headers=headers,
        max_size=1 << 24,   # allow large audio frames
    ) as ws:
        # --- 1) Configure the session as an interpreter (first thing we send) ---
        await ws.send(json.dumps({"type": "session.update", "session": session_config}))

        audio_out = bytearray()   # collect translated audio (PCM16) to play/save later

        async def send_audio():
            # Stream input frames, then close the turn and ask for a response.
            async for chunk in audio_chunks:
                await ws.send(json.dumps({
                    "type": "input_audio_buffer.append",
                    "audio": b64(chunk),
                }))
            # With server_vad these two are automatic, but sending them explicitly
            # makes the demo deterministic when we feed silence instead of speech.
            await ws.send(json.dumps({"type": "input_audio_buffer.commit"}))
            await ws.send(json.dumps({"type": "response.create"}))

        async def receive_events():
            async for message in ws:
                event = json.loads(message)
                etype = event.get("type", "")

                if etype == "session.created":
                    print("[session.created] connection live")
                elif etype == "session.updated":
                    print("[session.updated] interpreter config applied")
                elif etype == "input_audio_buffer.speech_started":
                    print("[vad] speech started")
                elif etype == "input_audio_buffer.speech_stopped":
                    print("[vad] speech stopped")
                elif etype == "conversation.item.input_audio_transcription.completed":
                    print(f"[heard] {event.get('transcript', '')!r}")
                elif etype == "response.audio_transcript.delta":
                    # Live caption of the spoken translation.
                    print(event.get("delta", ""), end="", flush=True)
                elif etype == "response.audio.delta":
                    # Streamed PCM16 audio of the translation -> buffer it.
                    audio_out.extend(base64.b64decode(event["delta"]))
                elif etype == "response.done":
                    print(f"\n[response.done] received {len(audio_out)} bytes of audio")
                    usage = event.get("response", {}).get("usage")
                    if usage:
                        print(f"[usage] {usage}")
                    break   # one turn done; exit the receive loop
                elif etype == "error":
                    print("\n[error]", json.dumps(event, indent=2))
                    break

        # Run send + receive concurrently; stop once the response completes.
        await asyncio.gather(send_audio(), receive_events())
        return bytes(audio_out)


async def _smoke_test_silence(num_chunks: int = 5):
    """Feed a few synthetic silence frames so the protocol runs without a mic."""
    async def silence_source():
        for _ in range(num_chunks):
            yield synthetic_silence_chunk()
            await asyncio.sleep(0.01)   # pace it like a real mic would
    return await realtime_interpreter(silence_source())


# In a notebook (already-running event loop) use `await`; from a script use asyncio.run().
# Uncomment to run the mic-free smoke test once your key + Realtime model are enabled:
#
#     audio = await _smoke_test_silence()        # in Jupyter
#     # audio = asyncio.run(_smoke_test_silence())  # in a .py script
#
print("Realtime interpreter loop defined. Run _smoke_test_silence() to exercise the protocol mic-free.")

## 4. Going live: real microphone capture and speaker playback

To turn the smoke test into a working interpreter, replace `synthetic_silence_chunk()` with real microphone frames and play `response.audio.delta` chunks through the speaker. We use **`sounddevice`** (a thin wrapper over PortAudio) for both directions.

Two details that trip people up:

- **Sample rate must match the session.** The Realtime session expects PCM16 at 24 kHz mono. Open the input/output streams at the same rate so you don't resample by accident.
- **Bridge the callback thread to asyncio.** `sounddevice` calls your audio callback on a separate thread. Push frames onto an `asyncio.Queue` (via `loop.call_soon_threadsafe`) so the async send loop can consume them safely.

The code below is complete — it needs only mic/speaker hardware and the Realtime model enabled. With `server_vad` turn detection you don't manually commit: just keep streaming mic frames and the server decides when the speaker paused and responds.

In [ ]:
import asyncio
import json
import base64
import numpy as np
import sounddevice as sd   # pip install sounddevice numpy

BLOCK_MS = 100
BLOCK_FRAMES = int(SAMPLE_RATE * BLOCK_MS / 1000)


async def mic_frames(stop_event: asyncio.Event):
    """Async generator of live PCM16 mic frames.

    sounddevice's InputStream calls `callback` on its own thread; we hand frames
    to the event loop via a threadsafe queue so the async send loop can await them.
    """
    loop = asyncio.get_running_loop()
    queue: asyncio.Queue[bytes] = asyncio.Queue()

    def callback(indata, frames, time_info, status):
        if status:
            print("[mic status]", status)
        # indata is float32 in [-1, 1]; convert to PCM16 little-endian bytes.
        pcm16 = (np.clip(indata[:, 0], -1.0, 1.0) * 32767).astype("<i2").tobytes()
        loop.call_soon_threadsafe(queue.put_nowait, pcm16)

    stream = sd.InputStream(
        samplerate=SAMPLE_RATE, channels=1, dtype="float32",
        blocksize=BLOCK_FRAMES, callback=callback,
    )
    with stream:
        while not stop_event.is_set():
            yield await queue.get()


class SpeakerPlayer:
    """Plays streamed PCM16 audio deltas through the default output device."""

    def __init__(self, sample_rate: int = SAMPLE_RATE):
        self.stream = sd.OutputStream(samplerate=sample_rate, channels=1, dtype="int16")
        self.stream.start()

    def play(self, pcm16_bytes: bytes):
        samples = np.frombuffer(pcm16_bytes, dtype="<i2")
        self.stream.write(samples)

    def close(self):
        self.stream.stop()
        self.stream.close()


async def live_interpreter(seconds: float = 30.0):
    """Run the interpreter live for `seconds` using the real mic + speaker.

    Reuses realtime_interpreter()'s protocol logic but wires in real audio I/O.
    NOTE: requires microphone + speaker hardware and the Realtime model enabled.
    """
    stop_event = asyncio.Event()
    player = SpeakerPlayer()

    headers = {
        "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
        "OpenAI-Beta": "realtime=v1",   # [unverified]
    }

    async with websockets.connect(
        f"{REALTIME_URL}?model={REALTIME_MODEL}",
        extra_headers=headers, max_size=1 << 24,
    ) as ws:
        await ws.send(json.dumps({"type": "session.update", "session": session_config}))

        async def send_audio():
            # Under server_vad we just keep streaming; the server segments turns.
            async for chunk in mic_frames(stop_event):
                await ws.send(json.dumps({
                    "type": "input_audio_buffer.append",
                    "audio": base64.b64encode(chunk).decode("ascii"),
                }))

        async def receive_events():
            async for message in ws:
                event = json.loads(message)
                etype = event.get("type", "")
                if etype == "response.audio_transcript.delta":
                    print(event.get("delta", ""), end="", flush=True)
                elif etype == "response.audio.delta":
                    player.play(base64.b64decode(event["delta"]))   # speak it
                elif etype == "error":
                    print("\n[error]", event)

        async def timer():
            await asyncio.sleep(seconds)
            stop_event.set()

        try:
            await asyncio.gather(send_audio(), receive_events(), timer())
        finally:
            player.close()


# Run a 30-second live interpreter session (needs mic + speaker):
#     await live_interpreter(30)          # in Jupyter
#     # asyncio.run(live_interpreter(30)) # in a .py script
print("Live mic/speaker interpreter defined. Call live_interpreter(30) with audio hardware to translate in real time.")

## 5. Where to take it next

- **Barge-in / interruption.** A real interpreter lets the speaker talk over the model. Watch for `input_audio_buffer.speech_started` while a response is playing and send a `response.cancel` to cut the current output.
- **Latency tuning.** Translation is not a deep-reasoning task — keep effort minimal and the voice fast so the round trip stays conversational. Smaller mic blocks lower TTFT at the cost of more WebSocket messages.
- **Both directions on one session.** Because the instructions say "translate EN→PT and PT→EN," a single session handles a two-person conversation; no need to detect language yourself.
- **Robustness.** Add reconnect-with-backoff on the WebSocket and always handle the `error` event — Realtime sessions can drop, and a production interpreter should recover silently.

This is the same WebSocket transport introduced in `00-introduction-openai-responses-api.ipynb`, specialized for bidirectional audio.